# trace_io.py —— 轨迹数据 npz 读写契约

`trace_io.py` 是信道生成（`sionna_traces.py`）与下游（env / 基线 / 评估）之间的**唯一数据接口**。一旦轨迹存成 npz，下游代码只读文件、绝不 import sionna——这是项目架构的铁律，保证：

- 信道建模变更（换模型、换种子）不污染策略代码；
- 策略代码可以在无 sionna / 无 GPU 的环境运行；
- 实验复现只需 npz 文件 + config 元数据。

## 数据契约

一个 npz 文件含三个键：

| 键 | 形状 | dtype | 含义 |
|---|---|---|---|
| `g` | $[N, T]$ | float32 | 每条轨迹、每时隙的**总功率增益**（线性），已含路径损耗 + 阴影 + 小尺度 |
| `d` | $[T]$ | float32 | 各时隙车-节点距离 $[\mathrm{m}]$，**同速度所有轨迹共享** |
| `meta` | 标量字符串 | np.array(str) | JSON 字符串，全部场景元数据（载频、T、dt、种子、TDL 参数…）|

**关键约定**：

- $g_t$ 是**线性功率增益**（不是 dB）：$g_t = 10^{-L_{\mathrm{fs}}(d_t)/10}\cdot 10^{\xi/10}\cdot g_{\mathrm{small}}(t)$
- 进入香农公式时直接用：$r_t = B\log_2(1 + p_t\, g_t / N)\,\Delta t$
- $d$ 之所以单独存而不并入 $g$，是因为：下游可能需要 $d$ 来画图、验算 FSPL、或做几何相关分析；它和 $g$ 是**因果同源**但语义不同的两个量。

## 存储格式选择

### 为什么用 npz 而不是 HDF5 / parquet

| 需求 | npz 是否满足 |
|---|---|
| 存多维 float32 数组 | ✅ 原生支持 |
| 存标量元数据 | ✅ 用 `np.array(json.dumps(meta))` 包一层 |
| 压缩 | ✅ `savez_compressed`（zlib）|
| 无外部依赖 | ✅ 只需 numpy |
| 跨平台 | ✅ npz 是 zip 容器，跨平台 |
| 随机访问单个轨迹 | ⚠️ 需全加载（但数据量小，无所谓）|

本项目数据量级：3 速度 × 1100 轨迹 × 1000 时隙 × 4 字节 ≈ 13 MB（压缩后 ~4 MB），npz 完全够用。

### meta 为什么要 JSON 字符串包一层

`np.savez` 不直接支持 dict，所以把 meta 序列化成 JSON 字符串再包成 0-d ndarray。读取时 `json.loads(str(z["meta"]))` 还原。

好处：

- meta 是**自描述**的：任何拿到 npz 的人都能读出场景参数，不需要翻代码；
- 字段可扩展：新增参数只改 dict，不动 I/O 代码；
- `ensure_ascii=False`：中文注释保留，npz 跨环境不乱码。

## I/O 函数

### `save_traces(path, g, d, meta)`

- 入口：信道生成脚本调用
- 强制 `float32`：避免 float64 双倍存储体积；float32 的 ~7 位有效数字对功率增益足够（增益跨 7 个数量级时，对数域才需要高精度——而 $g$ 存的是线性值，下游用 dB 时再转）
- **不在这里做 dB 转换**：保持线性，让下游（注水/香农公式）直接用，避免"存 dB 还是线性"的歧义

### `load_traces(path) -> (g, d, meta)`

- 入口：env / 基线 / 评估的唯一数据入口
- `allow_pickle=False`：安全默认，禁止反序列化任意对象（虽然 meta 是字符串，但 numpy 默认 pickle 会有风险）
- 返回 `(g, d, meta_dict)`，调用方一行解包

### float32 的一个已知陷阱

注水求解器在 `np.float32` 下做二分时，停止判据会被 float32 的 ~1e-6 误差污染，导致预算微微超出（相对 ~1e-7）。`baselines/power_alloc.py` 的修复方法是入口强制转 `float64`——这是**下游侧的防御**，不改变 npz 的存储契约。

In [1]:
# 演示：读写一条轨迹，验证契约一致性
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from channel.trace_io import save_traces, load_traces
import numpy as np

# 构造一条假轨迹
g = np.random.rand(5, 100).astype(np.float32) * 1e-6
d = np.linspace(20, 30, 100, dtype=np.float32)
meta = {"speed_kmh": 120, "T": 100, "dt_s": 1e-3, "note": "demo"}

# 写 → 读
save_traces('/tmp/_trace_io_demo.npz', g, d, meta)
g2, d2, m2 = load_traces('/tmp/_trace_io_demo.npz')

assert g2.dtype == np.float32 and g2.shape == (5, 100)
assert np.allclose(g, g2) and np.allclose(d, d2)
assert m2 == meta
print('契约验证通过 ✓')
print(f'g: shape={g2.shape}, dtype={g2.dtype}')
print(f'd: shape={d2.shape}, range=[{d2.min():.1f}, {d2.max():.1f}] m')
print(f'meta: {m2}')
os.remove('/tmp/_trace_io_demo.npz')


契约验证通过 ✓
g: shape=(5, 100), dtype=float32
d: shape=(100,), range=[20.0, 30.0] m
meta: {'speed_kmh': 120, 'T': 100, 'dt_s': 0.001, 'note': 'demo'}
